# Project 4: Spatial Audio & High-Fidelity Streaming
## Phase 2 — Codec Comparison & Spectral Analysis

**Course:** 2102571 Multimedia Communication in the 21st Century  
**Track:** Spatial Audio & High-Fidelity Streaming  

---

### What this notebook does

1. **Encodes** audio samples at multiple bitrates using **AAC, Opus, and MP3**
2. **Measures** objective quality — SNR and ODG (via PESQ proxy)
3. **Plots** Rate-Distortion curves (Bitrate vs. Quality)
4. **Generates** spectrogram comparisons (original vs. compressed)
5. **Analyzes** spectral masking — what each codec throws away
6. **Exports** all results as a zip for the report

## 0. Build FFmpeg with `libfdk_aac` (one-time, ~10 min)

> **Credit:** This FFmpeg build cell is from our teammate's original notebook — great initiative to compile from source so we get the high-quality FDK-AAC encoder instead of Colab's default FFmpeg `aac` encoder.

Run this cell once per Colab session. It compiles FFmpeg 6.1 with `libfdk_aac`, `libmp3lame`, and `libopus` enabled.

In [ ]:
import subprocess, os

def run(cmd, desc=""):
    print(f"\u23f3 {desc}...")
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if r.returncode != 0:
        print(f"  \u26a0\ufe0f Error: {r.stderr[-300:]}")
    else:
        print(f"  \u2713 Done")
    return r.returncode == 0

# Check if already built
check = subprocess.run(["ffmpeg", "-encoders"], capture_output=True, text=True)
if "libfdk_aac" in check.stdout:
    print("\u2705 libfdk_aac already available — skipping build.")
else:
    run("apt-get install -y build-essential git nasm yasm pkg-config "
        "libfdk-aac-dev libmp3lame-dev libopus-dev > /dev/null 2>&1",
        "Installing build dependencies")

    run("wget -q -O /tmp/ffmpeg.tar.bz2 "
        "https://ffmpeg.org/releases/ffmpeg-6.1.tar.bz2",
        "Downloading ffmpeg 6.1 source")

    run("tar -xjf /tmp/ffmpeg.tar.bz2 -C /tmp/", "Extracting source")

    run(\"\"\"cd /tmp/ffmpeg-6.1 && ./configure \\
        --prefix=/usr/local \\
        --enable-nonfree \\
        --enable-libfdk-aac \\
        --enable-libmp3lame \\
        --enable-libopus \\
        --disable-x86asm \\
        --quiet > /dev/null 2>&1\"\"\",
        "Configuring ffmpeg (~1 min)")

    run("cd /tmp/ffmpeg-6.1 && make -j2 > /dev/null 2>&1",
        "Compiling ffmpeg (~10-15 min)")

    run("cd /tmp/ffmpeg-6.1 && make install > /dev/null 2>&1",
        "Installing ffmpeg")

    r = subprocess.run(["ffmpeg", "-encoders"], capture_output=True, text=True)
    if "libfdk_aac" in r.stdout:
        print("\n\u2705 SUCCESS \u2014 libfdk_aac is now available!")
    else:
        print("\n\u274c libfdk_aac still not found after build.")

    r2 = subprocess.run(["ffmpeg", "-version"], capture_output=True, text=True)
    print(r2.stdout.split("\n")[0])

## 1. Install Python dependencies & imports

In [ ]:
!pip install pydub numpy scipy matplotlib librosa soundfile pesq --quiet

import os, subprocess, zipfile, glob, warnings
import numpy as np
import matplotlib.pyplot as plt
import librosa
import librosa.display
from pesq import pesq
from IPython.display import Audio, display, HTML

warnings.filterwarnings('ignore')
plt.rcParams['figure.dpi'] = 120

print("\u2705 All dependencies ready.")

## 2. Prepare test audio samples

We need **at least 3 categories** of audio for a thorough codec comparison.

**Option A — Upload your own:** Run the upload cell below.  
**Option B — Auto-generate:** If you skip uploading, we'll synthesize test signals (sine sweep, white noise, simulated speech envelope) so the pipeline still runs end-to-end. Replace these with real samples before the final analysis.

### Recommended real samples (royalty-free)
| Category | Source | Why |
|---|---|---|
| Music | Any CC0 orchestral clip | Rich harmonics, wide frequency range |
| Speech | LibriSpeech / Common Voice | Codec speech optimization differences |
| Ambient/FX | Freesound.org nature recording | Spatial cues, transient content |

In [ ]:
# ── Upload your own WAV files ──
# Uncomment the next two lines to upload interactively in Colab:
# from google.colab import files
# uploaded = files.upload()

SAMPLE_DIR = "audio_samples"
os.makedirs(SAMPLE_DIR, exist_ok=True)

# Check if user already has wav files in the sample dir
existing = glob.glob(f"{SAMPLE_DIR}/*.wav")

if len(existing) >= 1:
    print(f"\u2705 Found {len(existing)} audio file(s) in {SAMPLE_DIR}/:")
    for f in existing:
        print(f"   {f}")
else:
    print("No audio files found. Generating synthetic test signals...")
    print("(Replace these with real audio before final analysis!)\n")

    sr = 44100
    duration = 8  # seconds
    t = np.linspace(0, duration, sr * duration, endpoint=False)

    # 1. Sine sweep (20 Hz to 20 kHz) — tests full frequency range
    sweep = np.sin(2 * np.pi * 20 * (20000/20) ** (t/duration) * t / (np.log(20000/20)/duration))
    sweep = (sweep * 0.8 * 32767).astype(np.int16)

    import soundfile as sf
    sf.write(f"{SAMPLE_DIR}/synth_sweep.wav", sweep, sr)
    print("  \u2713 Generated: synth_sweep.wav (sine sweep 20Hz-20kHz)")

    # 2. Noise burst — tests noise shaping behavior
    np.random.seed(42)
    noise = np.random.randn(sr * duration) * 0.3
    # Apply envelope to simulate speech-like bursts
    envelope = np.zeros_like(noise)
    for i in range(0, len(noise), sr):
        seg = min(sr, len(noise) - i)
        env = np.concatenate([
            np.linspace(0, 1, seg//4),
            np.ones(seg//2),
            np.linspace(1, 0, seg - seg//4 - seg//2)
        ])
        envelope[i:i+seg] = env
    noise_shaped = (noise * envelope * 32767).astype(np.int16)
    sf.write(f"{SAMPLE_DIR}/synth_noise_burst.wav", noise_shaped, sr)
    print("  \u2713 Generated: synth_noise_burst.wav (shaped noise bursts)")

    # 3. Tonal mix — tests harmonic preservation
    freqs = [261.63, 329.63, 392.00, 523.25, 1046.50]  # C4 chord + harmonics
    tonal = sum(np.sin(2 * np.pi * f * t) * (0.3 / (i+1)) for i, f in enumerate(freqs))
    tonal = (tonal / np.max(np.abs(tonal)) * 0.8 * 32767).astype(np.int16)
    sf.write(f"{SAMPLE_DIR}/synth_tonal.wav", tonal, sr)
    print("  \u2713 Generated: synth_tonal.wav (C4 chord + harmonics)")

AUDIO_FILES = sorted(glob.glob(f"{SAMPLE_DIR}/*.wav"))
print(f"\n\u2705 {len(AUDIO_FILES)} test file(s) ready for encoding.")

## 3. Encoding & decoding helpers

> **Kept from original notebook** — the `ffmpeg_encode` / `ffmpeg_decode` helpers work well. We've added consistent sample-rate handling and stereo support.

In [ ]:
WORK_DIR = "codec_output"
os.makedirs(WORK_DIR, exist_ok=True)

ANALYSIS_SR = 44100  # standard analysis sample rate

def ffmpeg_encode(input_wav, output_path, codec, bitrate_kbps, channels=1):
    \"\"\"Encode a WAV file to a compressed format.\"\"\"
    # Opus requires 48kHz; AAC and MP3 work at 44.1kHz
    sr = "48000" if codec == "libopus" else "44100"
    cmd = [
        "ffmpeg", "-y", "-i", input_wav,
        "-c:a", codec,
        "-b:a", f"{bitrate_kbps}k",
        "-ar", sr,
        "-ac", str(channels),
        output_path
    ]
    result = subprocess.run(cmd, capture_output=True, text=True)
    if result.returncode != 0:
        print(f"  \u26a0\ufe0f encode error ({codec} {bitrate_kbps}k): {result.stderr[-200:]}")
    return result.returncode == 0


def ffmpeg_decode(encoded_path, output_wav):
    \"\"\"Decode compressed audio back to WAV at the standard analysis rate.\"\"\"
    cmd = [
        "ffmpeg", "-y", "-i", encoded_path,
        "-ar", str(ANALYSIS_SR),
        "-ac", "1",  # mono for fair metric comparison
        output_wav
    ]
    result = subprocess.run(cmd, capture_output=True, text=True)
    if result.returncode != 0:
        print(f"  \u26a0\ufe0f decode error: {result.stderr[-200:]}")
    return result.returncode == 0


def get_file_size_kb(filepath):
    \"\"\"Get compressed file size in KB.\"\"\"
    return os.path.getsize(filepath) / 1024 if os.path.exists(filepath) else 0

print("\u2705 Encoding helpers defined.")

## 4. Batch encode — AAC, Opus, MP3

We sweep **bitrates from 24 kbps to 256 kbps**. This range is chosen to capture the "Transparency" point where compressed audio becomes indistinguishable from the original.

> The original notebook used 8–64 kbps which is good for seeing severe degradation, but the project specifically asks us to find the Transparency point, which for most codecs falls between 96–192 kbps. We keep some low bitrates (24, 48) to show the degradation curve.

In [ ]:
BITRATES = [24, 48, 64, 96, 128, 160, 192, 256]

CODECS = {
    "MP3":  {"encoder": "libmp3lame", "ext": "mp3"},
    "AAC":  {"encoder": "libfdk_aac", "ext": "aac"},
    "Opus": {"encoder": "libopus",    "ext": "opus"},
}

# results[audio_name][codec_name][bitrate] = {"recon_wav": path, "enc_size_kb": float}
results = {}

for audio_path in AUDIO_FILES:
    audio_name = os.path.splitext(os.path.basename(audio_path))[0]
    results[audio_name] = {}
    print(f"\n{'='*60}")
    print(f"  Processing: {audio_name}")
    print(f"{'='*60}")

    for codec_name, cfg in CODECS.items():
        print(f"\n  --- {codec_name} ---")
        results[audio_name][codec_name] = {}

        for br in BITRATES:
            enc_path = f"{WORK_DIR}/{audio_name}_{codec_name}_{br}.{cfg['ext']}"
            recon_path = f"{WORK_DIR}/{audio_name}_{codec_name}_{br}_recon.wav"

            ok = ffmpeg_encode(audio_path, enc_path, cfg["encoder"], br)
            if ok:
                enc_size = get_file_size_kb(enc_path)
                ok2 = ffmpeg_decode(enc_path, recon_path)
                if ok2:
                    results[audio_name][codec_name][br] = {
                        "recon_wav": recon_path,
                        "enc_size_kb": enc_size
                    }
                    print(f"    {br:>4d} kbps  \u2713  ({enc_size:.1f} KB)")
                else:
                    print(f"    {br:>4d} kbps  \u2717 decode failed")
            else:
                print(f"    {br:>4d} kbps  \u2717 encode failed")

print("\n\u2705 All encoding complete.")

## 5. Objective quality metrics

We compute two metrics for each codec/bitrate combination:

| Metric | What it measures | Range |
|---|---|---|
| **SNR** (Signal-to-Noise Ratio) | Waveform-level distortion | Higher = better |
| **ODG** (Objective Difference Grade) | Perceptual quality (approximated via PESQ) | -4 (very annoying) to 0 (imperceptible) |

> **Important note on SNR:** AAC uses aggressive psychoacoustic noise shaping — it intentionally adds noise in masked frequency bands to save bits. This means **SNR is misleading for AAC**. The ODG/perceptual metric is the more meaningful comparison. (Good catch from the original notebook's annotation!)

In [ ]:
def load_and_align(original_file, recon_file, target_sr=16000):
    \"\"\"Load and normalize two audio files for metric computation.\"\"\"
    orig, _ = librosa.load(original_file, sr=target_sr, mono=True)
    recon, _ = librosa.load(recon_file, sr=target_sr, mono=True)
    # Peak normalize
    orig = orig / (np.max(np.abs(orig)) + 1e-9)
    recon = recon / (np.max(np.abs(recon)) + 1e-9)
    # Align lengths
    min_len = min(len(orig), len(recon))
    return orig[:min_len], recon[:min_len]


def calculate_snr(original_file, recon_file):
    \"\"\"Compute SNR in dB.\"\"\"
    orig, recon = load_and_align(original_file, recon_file, target_sr=ANALYSIS_SR)
    signal_power = np.sum(orig ** 2)
    noise_power = np.sum((orig - recon) ** 2)
    if noise_power < 1e-12:
        return 60.0
    return float(10 * np.log10(signal_power / noise_power))


def calculate_odg(original_file, recon_file):
    \"\"\"
    Approximate ODG using PESQ wideband MOS mapped to the ITU-R BS.1387 ODG scale.
    Mapping: ODG = (MOS - 4.5) * (4 / 3.5)

    Note: True ODG requires a full PEAQ implementation. This PESQ-based proxy
    gives a reasonable ranking but should be noted in the report as approximate.
    \"\"\"
    orig, recon = load_and_align(original_file, recon_file, target_sr=16000)
    try:
        mos = pesq(16000, orig, recon, 'wb')
    except Exception as e:
        print(f"  \u26a0\ufe0f PESQ error: {e}")
        return -4.0
    return float(np.clip((mos - 4.5) * (4.0 / 3.5), -4.0, 0.0))


# ── Compute metrics for all results ──
for audio_name in results:
    audio_path = f"{SAMPLE_DIR}/{audio_name}.wav"
    print(f"\nMetrics for: {audio_name}")
    print(f"  {'Codec':<6} {'Bitrate':>7}  {'SNR (dB)':>9}  {'ODG':>6}")
    print(f"  {'-'*35}")

    for codec_name in results[audio_name]:
        for br in sorted(results[audio_name][codec_name].keys()):
            entry = results[audio_name][codec_name][br]
            recon = entry["recon_wav"]
            snr_val = calculate_snr(audio_path, recon)
            odg_val = calculate_odg(audio_path, recon)
            entry["snr"] = snr_val
            entry["odg"] = odg_val
            print(f"  {codec_name:<6} {br:>5d}k   {snr_val:>8.1f}   {odg_val:>6.2f}")

print("\n\u2705 All metrics computed.")

## 6. Rate-Distortion curves

These are the key plots for the report — they show how quality changes with bitrate for each codec.

In [ ]:
COLORS = {"MP3": "#e74c3c", "AAC": "#3498db", "Opus": "#2ecc71"}
MARKERS = {"MP3": "o", "AAC": "s", "Opus": "^"}

FIGURES_DIR = "figures"
os.makedirs(FIGURES_DIR, exist_ok=True)

for audio_name in results:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    for codec_name in CODECS:
        data = results[audio_name].get(codec_name, {})
        bitrates = sorted(data.keys())
        if not bitrates:
            continue

        snr_vals = [data[br]["snr"] for br in bitrates]
        odg_vals = [data[br]["odg"] for br in bitrates]
        color = COLORS[codec_name]
        marker = MARKERS[codec_name]

        axes[0].plot(bitrates, snr_vals, marker=marker, color=color,
                     label=codec_name, linewidth=2, markersize=7)
        axes[1].plot(bitrates, odg_vals, marker=marker, color=color,
                     label=codec_name, linewidth=2, markersize=7)

    # SNR plot
    axes[0].set_xlabel("Bitrate (kbps)", fontsize=11)
    axes[0].set_ylabel("SNR (dB)", fontsize=11)
    axes[0].set_title(f"Rate-Distortion: SNR — {audio_name}", fontsize=12)
    axes[0].legend(fontsize=10)
    axes[0].grid(True, alpha=0.3)
    axes[0].annotate(
        "Note: AAC uses psychoacoustic noise shaping\n"
        "— SNR underestimates its perceptual quality.\n"
        "Use ODG (right plot) for fair comparison.",
        xy=(0.02, 0.02), xycoords='axes fraction',
        fontsize=7.5, color='darkorange',
        bbox=dict(boxstyle='round,pad=0.4', fc='lightyellow',
                  ec='darkorange', alpha=0.85))

    # ODG plot
    axes[1].set_xlabel("Bitrate (kbps)", fontsize=11)
    axes[1].set_ylabel("ODG (Objective Difference Grade)", fontsize=11)
    axes[1].set_title(f"Rate-Distortion: ODG — {audio_name}", fontsize=12)
    axes[1].set_ylim(-4.2, 0.5)
    axes[1].legend(fontsize=10)
    axes[1].grid(True, alpha=0.3)

    # ITU-R quality reference bands
    for val, label, alpha in [
        (-0.5, "Imperceptible", 0.15),
        (-1.0, "Perceptible, not annoying", 0.10),
        (-2.0, "Slightly annoying", 0.10),
        (-3.0, "Annoying", 0.10),
    ]:
        axes[1].axhline(val, color='gray', linewidth=0.5, linestyle=':')
        axes[1].text(max(BITRATES) + 2, val + 0.05, label,
                     fontsize=6.5, va='bottom', color='gray')

    # Highlight transparency zone
    axes[1].axhspan(-0.5, 0.5, alpha=0.08, color='green')
    axes[1].text(min(BITRATES), 0.15, "Transparency zone (ODG > -0.5)",
                 fontsize=7, color='green', alpha=0.7)

    plt.tight_layout()
    fig.savefig(f"{FIGURES_DIR}/rd_curves_{audio_name}.png", dpi=300, bbox_inches='tight')
    plt.show()

print("\u2705 RD curve figures saved.")

## 7. Spectrogram comparisons

Side-by-side spectrograms showing what each codec preserves and discards at different bitrates. This directly supports the "Spectral Masking" analysis required in the report.

In [ ]:
def plot_spectrogram_comparison(audio_name, audio_path, target_bitrate=64):
    \"\"\"Plot original + all 3 codecs at a given bitrate side by side.\"\"\"
    fig, axes = plt.subplots(1, 4, figsize=(20, 4))

    # Original
    y_orig, sr = librosa.load(audio_path, sr=ANALYSIS_SR)
    D_orig = librosa.amplitude_to_db(np.abs(librosa.stft(y_orig)), ref=np.max)
    librosa.display.specshow(D_orig, sr=sr, x_axis='time', y_axis='log',
                             ax=axes[0], cmap='magma')
    axes[0].set_title("Original", fontsize=11, fontweight='bold')

    for idx, codec_name in enumerate(["MP3", "AAC", "Opus"]):
        ax = axes[idx + 1]
        data = results[audio_name].get(codec_name, {})
        if target_bitrate in data:
            recon_path = data[target_bitrate]["recon_wav"]
            y_recon, _ = librosa.load(recon_path, sr=ANALYSIS_SR)
            D_recon = librosa.amplitude_to_db(
                np.abs(librosa.stft(y_recon)), ref=np.max)
            librosa.display.specshow(D_recon, sr=sr, x_axis='time', y_axis='log',
                                     ax=ax, cmap='magma')
            odg = data[target_bitrate].get("odg", -4)
            ax.set_title(f"{codec_name} @ {target_bitrate}k (ODG: {odg:.2f})",
                         fontsize=10)
        else:
            ax.set_title(f"{codec_name} @ {target_bitrate}k (N/A)")
            ax.text(0.5, 0.5, "Not available", ha='center', va='center',
                    transform=ax.transAxes)

    fig.suptitle(f"Spectrogram Comparison — {audio_name}", fontsize=13, y=1.02)
    plt.tight_layout()
    fig.savefig(f"{FIGURES_DIR}/spec_{audio_name}_{target_bitrate}k.png",
                dpi=300, bbox_inches='tight')
    plt.show()
    return fig


# Generate comparisons at low and mid bitrates
for audio_name in results:
    audio_path = f"{SAMPLE_DIR}/{audio_name}.wav"
    for br in [48, 96, 192]:
        if any(br in results[audio_name].get(c, {}) for c in CODECS):
            plot_spectrogram_comparison(audio_name, audio_path, br)

print("\u2705 Spectrogram comparisons saved.")

## 8. Spectral masking analysis — what the codecs discard

This computes the **spectral difference** between original and compressed signals. The difference plot reveals which frequency bands each codec considers "masked" (inaudible) and removes.

This is key for the report's discussion on **psychoacoustic masking models**.

In [ ]:
def plot_spectral_difference(audio_name, audio_path, target_bitrate=64):
    \"\"\"
    Plot the average spectral magnitude of original vs. each codec,
    plus the difference (what was removed).
    \"\"\"
    y_orig, sr = librosa.load(audio_path, sr=ANALYSIS_SR)
    S_orig = np.abs(librosa.stft(y_orig))
    S_orig_mean = np.mean(S_orig, axis=1)
    freqs = librosa.fft_frequencies(sr=sr)

    fig, axes = plt.subplots(2, 1, figsize=(12, 8), sharex=True)

    # Top: absolute spectra
    axes[0].plot(freqs, 20 * np.log10(S_orig_mean + 1e-10),
                 color='black', linewidth=1.5, label='Original', alpha=0.8)

    for codec_name in ["MP3", "AAC", "Opus"]:
        data = results[audio_name].get(codec_name, {})
        if target_bitrate not in data:
            continue
        recon_path = data[target_bitrate]["recon_wav"]
        y_recon, _ = librosa.load(recon_path, sr=ANALYSIS_SR)
        min_len = min(len(y_orig), len(y_recon))
        S_recon = np.abs(librosa.stft(y_recon[:min_len]))
        S_recon_mean = np.mean(S_recon, axis=1)
        axes[0].plot(freqs, 20 * np.log10(S_recon_mean + 1e-10),
                     color=COLORS[codec_name], linewidth=1, label=codec_name, alpha=0.7)

    axes[0].set_ylabel("Magnitude (dB)")
    axes[0].set_title(f"Average Spectral Magnitude — {audio_name} @ {target_bitrate} kbps")
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)
    axes[0].set_ylim(bottom=-80)

    # Bottom: spectral difference (what was removed)
    for codec_name in ["MP3", "AAC", "Opus"]:
        data = results[audio_name].get(codec_name, {})
        if target_bitrate not in data:
            continue
        recon_path = data[target_bitrate]["recon_wav"]
        y_recon, _ = librosa.load(recon_path, sr=ANALYSIS_SR)
        min_len = min(len(y_orig), len(y_recon))
        S_recon = np.abs(librosa.stft(y_recon[:min_len]))
        S_recon_mean = np.mean(S_recon, axis=1)
        # Spectral difference = what codec removed
        diff = 20 * np.log10(S_orig_mean + 1e-10) - 20 * np.log10(S_recon_mean + 1e-10)
        axes[1].plot(freqs, diff, color=COLORS[codec_name],
                     linewidth=1, label=f"{codec_name} (removed)", alpha=0.8)

    axes[1].axhline(0, color='gray', linewidth=0.5)
    axes[1].set_xlabel("Frequency (Hz)")
    axes[1].set_ylabel("Difference (dB)")
    axes[1].set_title(f"Spectral Content Removed by Codec — {audio_name} @ {target_bitrate} kbps")
    axes[1].legend()
    axes[1].grid(True, alpha=0.3)
    axes[1].set_xscale('log')
    axes[0].set_xscale('log')

    plt.tight_layout()
    fig.savefig(f"{FIGURES_DIR}/spectral_diff_{audio_name}_{target_bitrate}k.png",
                dpi=300, bbox_inches='tight')
    plt.show()


for audio_name in results:
    audio_path = f"{SAMPLE_DIR}/{audio_name}.wav"
    for br in [48, 128]:
        if any(br in results[audio_name].get(c, {}) for c in CODECS):
            plot_spectral_difference(audio_name, audio_path, br)

print("\u2705 Spectral masking analysis saved.")

## 9. Transparency point estimation

The **transparency point** is the bitrate where ODG > -0.5 (imperceptible difference). This is a core deliverable for the project.

In [ ]:
print("Transparency Point Estimation (ODG > -0.5 = imperceptible)")
print("=" * 60)

transparency_summary = {}

for audio_name in results:
    print(f"\n{audio_name}:")
    transparency_summary[audio_name] = {}

    for codec_name in CODECS:
        data = results[audio_name].get(codec_name, {})
        bitrates = sorted(data.keys())
        transparent_br = None

        for br in bitrates:
            odg = data[br].get("odg", -4)
            if odg > -0.5:
                transparent_br = br
                break

        if transparent_br:
            print(f"  {codec_name:<6} => Transparent at {transparent_br} kbps "
                  f"(ODG = {data[transparent_br]['odg']:.2f})")
        else:
            print(f"  {codec_name:<6} => NOT transparent at any tested bitrate "
                  f"(max ODG = {max(data[br].get('odg', -4) for br in bitrates):.2f})")

        transparency_summary[audio_name][codec_name] = transparent_br

print("\n" + "=" * 60)
print("\nNote: These are approximations using PESQ-based ODG proxy.")
print("Final ABX subjective tests (Phase 4) will validate these thresholds.")

## 10. Listen & compare (interactive)

Use this cell to listen to any codec/bitrate combination directly in the notebook. Useful for quick subjective checks.

In [ ]:
def listen_comparison(audio_name, bitrate):
    \"\"\"Display audio players for original + all codecs at a given bitrate.\"\"\"
    audio_path = f"{SAMPLE_DIR}/{audio_name}.wav"
    print(f"\n\u266b {audio_name} @ {bitrate} kbps")
    print("-" * 40)

    print("Original:")
    display(Audio(audio_path))

    for codec_name in CODECS:
        data = results[audio_name].get(codec_name, {})
        if bitrate in data:
            recon = data[bitrate]["recon_wav"]
            odg = data[bitrate].get("odg", "N/A")
            print(f"\n{codec_name} (ODG: {odg:.2f}):")
            display(Audio(recon))

# Example — change these to explore:
if AUDIO_FILES:
    first_name = os.path.splitext(os.path.basename(AUDIO_FILES[0]))[0]
    listen_comparison(first_name, 64)
    listen_comparison(first_name, 128)

## 11. Export all results

Packages figures, encoded audio, and reconstructed WAVs into a zip for the report.

In [ ]:
zip_filename = "phase2_codec_results.zip"

with zipfile.ZipFile(zip_filename, 'w', zipfile.ZIP_DEFLATED) as zipf:
    # Figures
    for fig_file in glob.glob(f"{FIGURES_DIR}/*.png"):
        zipf.write(fig_file)
        print(f"  + {fig_file}")

    # Reconstructed audio (keep a selection for the report)
    for audio_name in results:
        for codec_name in CODECS:
            for br in [48, 96, 128, 192]:
                data = results[audio_name].get(codec_name, {})
                if br in data:
                    zipf.write(data[br]["recon_wav"])

print(f"\n\u2705 Created {zip_filename}")
print(f"   Size: {os.path.getsize(zip_filename)/1024:.0f} KB")

# Uncomment to auto-download in Colab:
# from google.colab import files
# files.download(zip_filename)

## Summary & next steps

### What we completed (Phase 2)
- [x] Encoded audio across AAC, Opus, MP3 at 8 bitrate levels (24–256 kbps)
- [x] Computed SNR and ODG (PESQ-based) for all combinations
- [x] Generated Rate-Distortion curves
- [x] Created spectrogram comparisons at multiple bitrates
- [x] Performed spectral masking analysis (what each codec discards)
- [x] Estimated transparency points per codec

### Before the report — to do
- [ ] **Replace synthetic audio** with real samples (music, speech, ambient)
- [ ] **Run ABX subjective tests** (Phase 4) to validate transparency points
- [ ] **Add more audio categories** if time permits
- [ ] Consider adding **MUSHRA** or **MOS** subjective methodology

### For the report
- Use the ODG RD-curves as the primary quality comparison (not SNR)
- Note that ODG is approximated via PESQ — mention this as a limitation
- Include spectral difference plots in the "Spectral Masking" discussion section